# 13(f) securities pipeline

Builds the CUSIP reference tables from the SEC *Official List of Section 13(f) Securities*:
quarterly CSV files → `input_1.f13_securities` (one row per list entry) → `clean_up_1.asset` (one row per CUSIP). Every block shows the database feedback
(status message, affected rows, duration, result set) and is written to the run log
`logs/f13_securities/<RUN_ID>/`, so runs can be compared later (section 6). Execution and logging code lives
in `src/pipeline_helpers.py`.

The CSV files come from `Database/13f_securities` (`main.py`): SEC PDF lists → text extraction (2004+) or
OCR (1996–2003, issues in `data/ocr_issues.tsv`) → fixed-width TXT → `data/csv/13flistYYYYqN.csv`,
1996 Q1 – 2020 Q4.

| Section | Phase label | What happens |
|---|---|---|
| 0.2 Database connection | `connection` | read-only check, then working connection |
| 1. Securities table | `securities_create` | `f13_securities_setup.sql`, CUSIP check-digit function |
| 2. Load CSV files | `securities_load` | one transaction per file (replaces `load_13f_securities.py`) |
| 3. Securities profile | `securities_profile` | totals, rows per year, duplicates, invalid check digits |
| 4. Build asset table | `asset_build` | one row per valid CUSIP with first / last quarter |
| 5. Asset checks | `asset_checks` | counts, gaps in listing |

> **The cells modify the database.** `clean_up_1.asset` is dropped and rebuilt in section 4.

## 0. Setup

### 0.1 Configuration

* `CSV_DIR`: the quarterly list files
* `RECREATE_SECURITIES`: `True` drops and recreates the securities table before the load. Not needed for a
  normal re-run: the loader replaces each file's rows.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent                       # .../Database/Initialisation (notebooks run from notebooks/)
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "config")]

from pipeline_helpers import (F13ListLoader, PipelineRun, compare_runs, expected_mismatches,
                              load_logs, phase_durations, steps_table)
from local_config import DSN   # machine-specific, git-ignored (template: local_config.example.py)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

LOG_ROOT = ROOT / "logs" / "f13_securities"

SECURITIES_TABLE = "input_1.f13_securities"
ASSET_TABLE      = "clean_up_1.asset"

# Output of Database/13f_securities (SEC PDF lists -> TXT -> CSV)
CSV_DIR = ROOT.parent / "13f_securities" / "data" / "csv"

RECREATE_SECURITIES = True   # True = drop and recreate the securities table before the load

# The tables are small (<1M rows), so exact count(*) is cheap
p = PipelineRun(LOG_ROOT, dsn=DSN, max_rows=1000, exact_counts=True)
print("RUN_ID :", p.run_id)
print("LOG_DIR:", p.log_dir)

RUN_ID : 20261011_201543
LOG_DIR: <THESIS_DIR>/src/Clean_solution/Database/Initialisation/pipelines/logs/f13_securities/20261011_201543


### 0.2 Database connection

First a read-only check on a separate connection (`READ ONLY` transaction, rolled back): server, database,
user, schema privileges, current state of both tables. The working connection (autocommit) is opened only if
the check passes.

In [2]:
CONNECTION_OK = p.check_connection(DSN, schemas=["input_1", "clean_up_1"],
                                   tables=[SECURITIES_TABLE, ASSET_TABLE])

✅ **[connection] connection_check_server**: `ok` | 0.0s

> READ ONLY transaction, rolled back. Nothing committed.

,server_version,database,db_user,is_standby,txn_read_only,db_size,other_sessions
0,PostgreSQL 16.15 (Ubuntu 16.15-0ubuntu0.24.04....,Holdings_v2,postgres,False,on,53 GB,2


✅ **[connection] connection_check_schemas**: `ok`

,schema_name,schema_exists,can_create,can_use
0,input_1,True,True,True
1,clean_up_1,True,True,True


✅ **[connection] connection_check_tables**: `ok`

> Existing tables will be dropped and recreated by the pipeline.

,table_name,exists,unlogged,est_rows,total_size
0,input_1.f13_securities,True,False,856547,123 MB
1,clean_up_1.asset,True,False,67216,39 MB


In [3]:
assert CONNECTION_OK, "Connection check failed, see connection_check_server above"
p.connect()

Connected: user=postgres password=xxx dbname=Holdings_v2 host=localhost port=5432


## 1. Securities table

`input_1.f13_securities` holds one row per list entry (`f13_securities_setup.sql`). There is no unique key on
(`cusip`, `year`, `quarter`): some OCR'd lists contain the same CUSIP twice.

`input_1.cusip_check_digit(c8)` computes the 9th CUSIP character from the first 8 (same function as in the holdings pipeline). A listed CUSIP whose 9th character differs was misread by the OCR (section 3).

In [4]:
sec = p.stage("securities_create")

if RECREATE_SECURITIES:
    sec.run("drop_securities_table", f"DROP TABLE IF EXISTS {SECURITIES_TABLE};")

sec.run("create_securities_table", f"""
    CREATE TABLE IF NOT EXISTS {SECURITIES_TABLE} (
        id                 BIGSERIAL PRIMARY KEY,
        year               SMALLINT    NOT NULL,
        quarter            SMALLINT    NOT NULL CHECK (quarter BETWEEN 1 AND 4),
        cusip              VARCHAR(9)  NOT NULL,
        has_option         BOOLEAN,
        issuer_name        TEXT,
        issuer_description TEXT,
        status             CHAR(1),
        flag               CHAR(1),
        source_file        TEXT        NOT NULL
    );
""")

✅ **[securities_create] drop_securities_table**: `ok` | `DROP TABLE` | 0.0s

✅ **[securities_create] create_securities_table**: `ok` | `CREATE TABLE` | 0.0s

""


In [5]:
sec.run("idx_securities_cusip", f"""
    CREATE INDEX IF NOT EXISTS f13_securities_cusip_idx ON {SECURITIES_TABLE} (cusip);
""")
sec.run("idx_securities_year_quarter", f"""
    CREATE INDEX IF NOT EXISTS f13_securities_year_quarter_idx ON {SECURITIES_TABLE} (year, quarter);
""")

✅ **[securities_create] idx_securities_cusip**: `ok` | `CREATE INDEX` | 0.0s

✅ **[securities_create] idx_securities_year_quarter**: `ok` | `CREATE INDEX` | 0.0s

""


In [6]:
sec.run("create_cusip_check_digit", r"""
    -- CUSIP check digit (9th character) computed from cusip8.
    -- Returns NULL if cusip8 is NULL or not 8 valid characters.
    CREATE OR REPLACE FUNCTION input_1.cusip_check_digit(c8 text)
    RETURNS text
    LANGUAGE sql IMMUTABLE STRICT PARALLEL SAFE AS $$
      SELECT CASE WHEN c8 ~ '^[0-9A-Za-z*@#]{8}$' THEN
        ((10 - sum(v / 10 + v % 10) % 10) % 10)::text
      END
      FROM (
        SELECT CASE WHEN i % 2 = 0 THEN 2 * val ELSE val END AS v   -- even (1-based) positions doubled
        FROM (
          SELECT i,
                 CASE WHEN ch ~ '[0-9]'    THEN ch::int
                      WHEN ch ~ '[A-Za-z]' THEN ascii(upper(ch)) - 55   -- A=10 ... Z=35
                      WHEN ch = '*' THEN 36
                      WHEN ch = '@' THEN 37
                      ELSE 38 END AS val                                -- '#'
          FROM (SELECT i, substr(c8, i, 1) AS ch FROM generate_series(1, 8) i) s
        ) t
      ) d
    $$;
""")

# Sanity check: Apple 037833100 -> check digit '0'
chk_digit = sec.run("check_digit_sanity", "SELECT input_1.cusip_check_digit('03783310') AS digit;")
assert chk_digit.iloc[0, 0] == "0", "cusip_check_digit is wrong"

✅ **[securities_create] create_cusip_check_digit**: `ok` | `CREATE FUNCTION` | 0.0s

✅ **[securities_create] check_digit_sanity**: `ok` | `SELECT 1` | rows: **1** | 0.0s

,digit
0,0


## 2. Load the CSV files

Same rules as `f13_securities/load_13f_securities.py`:
* `year` / `quarter` from the file name (`13flist1997q1.csv` → 1997, 1)
* `has_option` `Y` / `N` → boolean, blank fields → `NULL`, rows without a CUSIP are skipped
* each file is loaded in its own transaction and **replaces** that file's previous rows, so re-running is safe
* files are read as Latin-1; the header must be `cusip,has_option,issuer_name,issuer_description,status,flag`

The preflight lists the files and the missing quarters (1997 Q4 has no list).

In [7]:
loader = F13ListLoader(p.stage("securities_load"), CSV_DIR, SECURITIES_TABLE)

loader.preflight()

✅ **[securities_load] preflight_csv_files**: `ok` | rows: **99**

> <PROJECTS_DIR>/Sec_data_gathering/13f_lists_pre2021/csv  |  missing quarters: 1997q4

,file,year,quarter,kb
0,13flist1996q1.csv,1996,1,429
1,13flist1996q2.csv,1996,2,433
2,13flist1996q3.csv,1996,3,438
3,13flist1996q4.csv,1996,4,446
4,13flist1997q1.csv,1997,1,448
...,...,...,...,...
94,13flist2019q4.csv,2019,4,385
95,13flist2020q1.csv,2020,1,395
96,13flist2020q2.csv,2020,2,400
97,13flist2020q3.csv,2020,3,408


True

In [8]:
loaded = loader.run()   # previous run: 99 files, 856,547 rows

✅ **[securities_load] load_csv_files**: `ok` | rows: **856,547** | 7.4s

> 99 files | skipped rows without CUSIP: 0

,file,year,quarter,in_csv,replaced,inserted,seconds,error
0,13flist1996q1.csv,1996,1,10480,0,10480,0.05,None
1,13flist1996q2.csv,1996,2,10654,0,10654,0.05,None
2,13flist1996q3.csv,1996,3,10796,0,10796,0.05,None
3,13flist1996q4.csv,1996,4,10992,0,10992,0.05,None
4,13flist1997q1.csv,1997,1,11046,0,11046,0.05,None
...,...,...,...,...,...,...,...,...
94,13flist2019q4.csv,2019,4,8866,0,8866,0.09,None
95,13flist2020q1.csv,2020,1,8880,0,8880,0.09,None
96,13flist2020q2.csv,2020,2,8972,0,8972,0.14,None
97,13flist2020q3.csv,2020,3,9134,0,9134,0.09,None


## 3. Securities profile

In [9]:
prof = p.stage("securities_profile")

prof.row_estimate(SECURITIES_TABLE, "securities")
prof.run("securities_totals", f"""
    SELECT count(*)                                          AS total,
           count(DISTINCT source_file)                       AS files,
           count(DISTINCT cusip)                             AS distinct_cusip,
           count(*) FILTER (WHERE length(cusip) <> 9)        AS cusip_not9,
           count(*) FILTER (WHERE has_option)                AS has_option,
           count(*) FILTER (WHERE has_option IS NULL)        AS has_option_null,
           count(*) - count(DISTINCT (cusip, year, quarter)) AS duplicate_in_quarter,
           count(*) FILTER (WHERE status = 'A')              AS status_added,
           count(*) FILTER (WHERE status = 'D')              AS status_deleted
    FROM {SECURITIES_TABLE};
""", note="Previous run: total 856,547 | files 99 | distinct_cusip 42,094 | cusip_not9 0 | "
          "has_option 362,283 | duplicate_in_quarter 399 | status A 30,121 | status D 30,585")

✅ **[securities_profile] securities_count_exact**: `ok` | `SELECT 1` | rows: **1** | 0.0s

,n
0,856547


✅ **[securities_profile] securities_totals**: `ok` | `SELECT 1` | rows: **1** | 2.0s

> Previous run: total 856,547 | files 99 | distinct_cusip 42,094 | cusip_not9 0 | has_option 362,283 | duplicate_in_quarter 399 | status A 30,121 | status D 30,585

,total,files,distinct_cusip,cusip_not9,has_option,has_option_null,duplicate_in_quarter,status_added,status_deleted
0,856547,99,42094,0,362283,0,399,30121,30585


,total,files,distinct_cusip,cusip_not9,has_option,has_option_null,duplicate_in_quarter,status_added,status_deleted
0,856547,99,42094,0,362283,0,399,30121,30585


In [10]:
prof.run("securities_per_year", f"""
    SELECT year,
           count(DISTINCT quarter)            AS quarters,
           count(*)                           AS rows,
           round(count(*)::numeric / count(DISTINCT quarter)) AS rows_per_quarter,
           count(*) FILTER (WHERE has_option) AS has_option
    FROM {SECURITIES_TABLE}
    GROUP BY year
    ORDER BY year;
""")

✅ **[securities_profile] securities_per_year**: `ok` | `SELECT 25` | rows: **25** | 0.1s

,year,quarters,rows,rows_per_quarter,has_option
0,1996,4,42922,10731,8730
1,1997,3,33125,11042,7244
2,1998,4,43601,10900,11681
3,1999,4,40646,10162,11620
4,2000,4,39242,9811,11661
5,2001,4,36408,9102,11965
6,2002,4,33765,8441,12488
7,2003,4,31710,7928,11707
8,2004,4,31594,7899,11697
9,2005,4,31965,7991,12396


,year,quarters,rows,rows_per_quarter,has_option
0,1996,4,42922,10731,8730
1,1997,3,33125,11042,7244
2,1998,4,43601,10900,11681
3,1999,4,40646,10162,11620
4,2000,4,39242,9811,11661
5,2001,4,36408,9102,11965
6,2002,4,33765,8441,12488
7,2003,4,31710,7928,11707
8,2004,4,31594,7899,11697
9,2005,4,31965,7991,12396


In [11]:
prof.run("securities_duplicates", f"""
    -- same CUSIP listed more than once in a quarter (OCR); only one is kept in the CUSIP table
    SELECT cusip, year, quarter, count(*) AS times,
           string_agg(DISTINCT coalesce(issuer_name, '(null)'), ' | ') AS issuer_names
    FROM {SECURITIES_TABLE}
    GROUP BY cusip, year, quarter
    HAVING count(*) > 1
    ORDER BY times DESC, year, quarter
    LIMIT 50;
""")

✅ **[securities_profile] securities_duplicates**: `ok` | `SELECT 50` | rows: **50** | 0.8s

,cusip,year,quarter,times,issuer_names
0,628546103,1996,1,3,MYLEX CORP
1,200300101,1996,1,3,COMCAST CORP | COMCAST CORP C
2,451841100,1996,1,3,ILLINOIS CENT CORP
3,895927101,1996,1,3,TRIARC COS INC C | TRIARC COS INC...
4,814208104,1997,2,3,SECURITY DYNAMICS TECHNOLOGIE
5,257661504,1999,2,3,DONALDSON LUFKIN & JENRETTE N
6,872879101,1999,2,3,TSI INTERNATIONAL S OFTWARE LT
7,743168106,1999,2,3,PROFIT RECOVERY GROUP INTL IN
8,371931106,1999,2,3,GENESYS TELE COMMUNICATION LAB
9,130209604,1999,2,3,CALIFORNIA FED BK FSB LOS ANG


,cusip,year,quarter,times,issuer_names
0,628546103,1996,1,3,MYLEX CORP
1,200300101,1996,1,3,COMCAST CORP | COMCAST CORP C
2,451841100,1996,1,3,ILLINOIS CENT CORP
3,895927101,1996,1,3,TRIARC COS INC C | TRIARC COS INC...
4,814208104,1997,2,3,SECURITY DYNAMICS TECHNOLOGIE
5,257661504,1999,2,3,DONALDSON LUFKIN & JENRETTE N
6,872879101,1999,2,3,TSI INTERNATIONAL S OFTWARE LT
7,743168106,1999,2,3,PROFIT RECOVERY GROUP INTL IN
8,371931106,1999,2,3,GENESYS TELE COMMUNICATION LAB
9,130209604,1999,2,3,CALIFORNIA FED BK FSB LOS ANG


### Check digit

`wrong_digit`: the 9th character is not the check digit of the first 8. `bad_characters`: the first 8 contain a
character a CUSIP cannot have (e.g. `?` from the OCR). `valid_twin`: a valid CUSIP with the same first 8
characters exists, so the 9th character was most likely misread. These CUSIPs are left out of the asset table.

In [12]:
prof.run("securities_check_digit", f"""
    WITH c AS (
        SELECT cusip, count(*) AS list_rows,
               input_1.cusip_check_digit(left(cusip, 8)) AS digit
        FROM {SECURITIES_TABLE}
        WHERE length(cusip) = 9
        GROUP BY cusip
    ),
    valid8 AS (SELECT DISTINCT left(cusip, 8) AS c8 FROM c WHERE digit = right(cusip, 1))
    SELECT count(*)                                                        AS distinct_cusips,
           count(*) FILTER (WHERE digit = right(cusip, 1))                 AS valid,
           count(*) FILTER (WHERE digit <> right(cusip, 1))                AS wrong_digit,
           count(*) FILTER (WHERE digit IS NULL)                           AS bad_characters,
           count(*) FILTER (WHERE digit IS DISTINCT FROM right(cusip, 1)
                              AND left(cusip, 8) IN (SELECT c8 FROM valid8)) AS valid_twin,
           sum(list_rows) FILTER (WHERE digit IS DISTINCT FROM right(cusip, 1)) AS invalid_list_rows
    FROM c;
""", note="Previous run: 42,094 CUSIPs | valid 40,324 | wrong_digit 1,593 | bad_characters 177 | "
          "valid_twin 753 | invalid_list_rows 2,164 of 856,547")

✅ **[securities_profile] securities_check_digit**: `ok` | `SELECT 1` | rows: **1** | 0.6s

> Previous run: 42,094 CUSIPs | valid 40,324 | wrong_digit 1,593 | bad_characters 177 | valid_twin 753 | invalid_list_rows 2,164 of 856,547

,distinct_cusips,valid,wrong_digit,bad_characters,valid_twin,invalid_list_rows
0,42094,40324,1593,177,753,2164


,distinct_cusips,valid,wrong_digit,bad_characters,valid_twin,invalid_list_rows
0,42094,40324,1593,177,753,2164


In [13]:
prof.run("repair_check_digit", f"""
    UPDATE {SECURITIES_TABLE} s
    SET cusip = left(s.cusip, 8) || input_1.cusip_check_digit(left(s.cusip, 8))
    WHERE length(s.cusip) = 9
      AND input_1.cusip_check_digit(left(s.cusip, 8)) IS NOT NULL
      AND input_1.cusip_check_digit(left(s.cusip, 8)) <> right(s.cusip, 1)
      AND right(s.cusip, 1) !~ '[0-9]';
""")

✅ **[securities_profile] repair_check_digit**: `ok` | `UPDATE 1162` | rows: **1,162** | 0.2s

""


## 4. Build the asset table

`clean_up_1.asset` has **one row per CUSIP** instead of one per CUSIP and quarter (`build_cusip.sql` built
`input_1.cusip` that way). A CUSIP does not change: it is listed from its first to its last quarter, so the table keeps:

* `first_year`, `first_quarter`, `last_year`, `last_quarter`: first and last list the CUSIP appears in
* `quarters_listed`: number of lists it appears in
* `quarters_missing`: lists between first and last where it is absent (1997 Q4 has no list and does not
  count). 0 = continuous; see section 5 for the causes of gaps
* `issuer_name`, `issuer_description`: from the latest list the CUSIP appears in (names can change, e.g. after
  a rebranding)

Only CUSIPs with a **valid check digit** are used (section 3); the others are OCR misreads.

No CALL / PUT rows are generated: options have no CUSIP of their own. 13F filings report an option position
under the **underlying's CUSIP** and mark it with `putCall` = `PUT` / `CALL`, so no holding matches a generated
option CUSIP (characters 7–8 = `90` / `95`). Option rows are identified by `putCall` and removed in
`1_holdings_pipeline.ipynb` (2.1). `has_option` stays in `input_1.f13_securities` as listed by the SEC.

In [14]:
build = p.stage("asset_build")

build.run("create_asset_table", f"""
    DROP TABLE IF EXISTS {ASSET_TABLE};

    CREATE TABLE {ASSET_TABLE} (
        cusip              VARCHAR(9)  PRIMARY KEY,
        issuer_name        TEXT,
        issuer_description TEXT,
        first_year         SMALLINT    NOT NULL,
        first_quarter      SMALLINT    NOT NULL CHECK (first_quarter BETWEEN 1 AND 4),
        last_year          SMALLINT    NOT NULL,
        last_quarter       SMALLINT    NOT NULL CHECK (last_quarter BETWEEN 1 AND 4),
        quarters_listed    SMALLINT    NOT NULL,
        quarters_missing   SMALLINT    NOT NULL,
        CHECK ((first_year, first_quarter) <= (last_year, last_quarter))
    );
""")

✅ **[asset_build] create_asset_table**: `ok` | `CREATE TABLE` | 0.0s

""


In [15]:
build.run("insert_underlying", f"""
    INSERT INTO {ASSET_TABLE}
        (cusip, issuer_name, issuer_description,
         first_year, first_quarter, last_year, last_quarter,
         quarters_listed, quarters_missing)
    WITH lists AS (
        -- every quarter that has a list, numbered 1, 2, 3, ... (1997 Q4 has no list)
        SELECT period, row_number() OVER (ORDER BY period) AS list_no
        FROM (SELECT DISTINCT year * 4 + quarter - 1 AS period FROM {SECURITIES_TABLE}) p
    ),
    a AS (
        SELECT cusip,
               min(year * 4 + quarter - 1)          AS first_period,
               max(year * 4 + quarter - 1)          AS last_period,
               count(DISTINCT year * 4 + quarter)   AS quarters_listed,
               -- name / description from the latest list the CUSIP appears in
               (array_agg(issuer_name        ORDER BY year DESC, quarter DESC, issuer_name))[1]        AS issuer_name,
               (array_agg(issuer_description ORDER BY year DESC, quarter DESC, issuer_description))[1] AS issuer_description
        FROM {SECURITIES_TABLE}
        WHERE length(cusip) = 9
          AND input_1.cusip_check_digit(left(cusip, 8)) = right(cusip, 1)   -- valid check digit only
        GROUP BY cusip
    )
    SELECT a.cusip, a.issuer_name, a.issuer_description,
           a.first_period / 4, a.first_period % 4 + 1,
           a.last_period  / 4, a.last_period  % 4 + 1,
           a.quarters_listed,
           (ll.list_no - lf.list_no + 1) - a.quarters_listed
    FROM a
    JOIN lists lf ON lf.period = a.first_period
    JOIN lists ll ON ll.period = a.last_period;
""", expected=40_593, note="one row per distinct CUSIP with a valid check digit (40,593 after the check-digit repair)")

✅ **[asset_build] insert_underlying**: `ok` | `INSERT 0 40593` | rows: **40,593** | expected: 40,593 (diff +0) | 8.5s

> one row per distinct CUSIP with a valid check digit (40,593 after the check-digit repair)

""


In [16]:
build.run("idx_asset_first_last", f"CREATE INDEX IF NOT EXISTS asset_first_last_idx ON {ASSET_TABLE} (first_year, first_quarter, last_year, last_quarter);")
build.run("analyze_asset", f"ANALYZE {ASSET_TABLE};")

✅ **[asset_build] idx_asset_first_last**: `ok` | `CREATE INDEX` | 0.0s

✅ **[asset_build] analyze_asset**: `ok` | `ANALYZE` | 0.1s

""


In [17]:
build.run("select_coverage", f"""
    WITH a AS (
    SELECT DISTINCT UPPER(TRIM(cusip)) AS cusip9,
                    LEFT(UPPER(TRIM(cusip)), 8) AS cusip8
    FROM clean_up_1.asset
),
h AS (
    SELECT DISTINCT UPPER(TRIM(cusip9)) AS cusip9,
                    LEFT(UPPER(TRIM(cusip9)), 8) AS cusip8
    FROM clean_up_1.holdings_clean
)
SELECT
    (SELECT COUNT(*) FROM h)                                         AS holdings_cusips,
    (SELECT COUNT(*) FROM a)                                         AS asset_cusips,
    (SELECT COUNT(*) FROM h WHERE cusip9 IN (SELECT cusip9 FROM a))  AS match_9,
    (SELECT COUNT(*) FROM h WHERE cusip8 IN (SELECT cusip8 FROM a))  AS match_8,
    (SELECT COUNT(*) FROM h
      WHERE cusip8 IN (SELECT cusip8 FROM a)
        AND cusip9 NOT IN (SELECT cusip9 FROM a))                    AS match_8_only,
    (SELECT COUNT(*) FROM h WHERE cusip8 NOT IN (SELECT cusip8 FROM a)) AS no_match;
""")

✅ **[asset_build] select_coverage**: `ok` | `SELECT 1` | rows: **1** | 15.3s

,holdings_cusips,asset_cusips,match_9,match_8,match_8_only,no_match
0,33610,40593,25807,33573,7766,37


,holdings_cusips,asset_cusips,match_9,match_8,match_8_only,no_match
0,33610,40593,25807,33573,7766,37


## 5. Asset checks

In [18]:
chk = p.stage("asset_checks")

chk.run("asset_summary", f"""
    SELECT count(*) AS assets,
           count(*) FILTER (WHERE quarters_missing > 0) AS with_gaps,
           min(first_year) AS first_year, max(last_year) AS last_year,
           round(avg(quarters_listed), 1) AS avg_quarters_listed
    FROM {ASSET_TABLE};
""", note="Previous test: 40,593 assets (2,551 with gaps)")

✅ **[asset_checks] asset_summary**: `ok` | `SELECT 1` | rows: **1** | 0.0s

> Previous test: 40,593 assets (2,551 with gaps)

,assets,with_gaps,first_year,last_year,avg_quarters_listed
0,40593,2551,1996,2020,21.1


,assets,with_gaps,first_year,last_year,avg_quarters_listed
0,40593,2551,1996,2020,21.1


### Gaps in listing

Most CUSIPs appear in one continuous run of lists. Gaps of 1–2 quarters are usually OCR misses; long gaps
are propably caused by rare appearence of given asset, changes in need to raport it or changes in one company (rebranding).

In [19]:
chk.run("asset_gap_distribution", f"""
    SELECT quarters_missing, count(*) AS assets
    FROM {ASSET_TABLE}
    GROUP BY quarters_missing
    ORDER BY quarters_missing
    LIMIT 30;
""", note="Previous test: 2,551 of 40,593 CUSIPs have at least one missing quarter")

✅ **[asset_checks] asset_gap_distribution**: `ok` | `SELECT 30` | rows: **30** | 0.0s

> Previous test: 2,551 of 40,593 CUSIPs have at least one missing quarter

,quarters_missing,assets
0,0,38042
1,1,1566
2,2,209
3,3,131
4,4,66
5,5,65
6,6,52
7,7,58
8,8,28
9,9,33


,quarters_missing,assets
0,0,38042
1,1,1566
2,2,209
3,3,131
4,4,66
5,5,65
6,6,52
7,7,58
8,8,28
9,9,33


In [20]:
chk.row_estimate(ASSET_TABLE, "asset")
chk.table_size(ASSET_TABLE, "asset")

✅ **[asset_checks] asset_count_exact**: `ok` | `SELECT 1` | rows: **1** | 0.0s

,n
0,40593


✅ **[asset_checks] asset_size**: `ok` | `SELECT 1` | rows: **1** | 0.0s

,total,heap,indexes
0,5304 kB,3304 kB,1968 kB


,total,heap,indexes
0,5304 kB,3304 kB,1968 kB


## 6. Run log analytics

Reads `steps.jsonl` from **all** runs in `logs/f13_securities/`.

In [21]:
logs = load_logs(LOG_ROOT)
steps_table(logs, p.run_id)

,seq,phase,step,status,statusmessage,rowcount,expected,diff,duration_s
116,1,connection,connection_check_server,ok,None,NaN,NaN,NaN,0.017
117,2,connection,connection_check_schemas,ok,None,NaN,NaN,NaN,NaN
118,3,connection,connection_check_tables,ok,None,NaN,NaN,NaN,NaN
119,4,securities_create,drop_securities_table,ok,DROP TABLE,-1.0,NaN,NaN,0.009
120,5,securities_create,create_securities_table,ok,CREATE TABLE,-1.0,NaN,NaN,0.010
121,6,securities_create,idx_securities_cusip,ok,CREATE INDEX,-1.0,NaN,NaN,0.004
122,7,securities_create,idx_securities_year_quarter,ok,CREATE INDEX,-1.0,NaN,NaN,0.003
123,8,securities_create,create_cusip_check_digit,ok,CREATE FUNCTION,-1.0,NaN,NaN,0.002
124,9,securities_create,check_digit_sanity,ok,SELECT 1,1.0,NaN,NaN,0.001
125,10,securities_load,preflight_csv_files,ok,None,99.0,NaN,NaN,NaN


In [22]:
# Row counts that differ from the previous run
expected_mismatches(logs, p.run_id)

,seq,phase,step,status,statusmessage,rowcount,expected,diff,duration_s


In [23]:
phase_durations(logs, p.run_id)

,seconds
phase,
connection,0.017
securities_create,0.029
securities_load,7.395
securities_profile,3.697
asset_build,23.994
asset_checks,0.015


In [24]:
compare_runs(logs)

rowcount                                                                                      duration_s                                  \
run_id                                         20261008_212211 20261009_103639 20261009_110933 20261009_115233 20261009_120226 20261011_201543 20261008_212126 20261008_212211 20261009_103639   
phase              step                                                                                                                                                                          
connection         connection_check_server                 NaN             NaN             NaN             NaN             NaN             NaN           0.016           0.012             NaN   
securities_create  drop_securities_table                  -1.0             NaN            -1.0             NaN            -1.0            -1.0             NaN           0.010             NaN   
                   create_securities_table                -1.0             NaN            -1.0             NaN            -1.0            -1.0             NaN           0.009             NaN   
                   idx_securities_cusip                   -1.0             NaN            -1.0             NaN            -1.0            -1.0             NaN           0.007             NaN   
                   idx_securities_year_quarter            -1.0             NaN            -1.0             NaN            -1.0            -1.0             NaN           0.004             NaN   
                   create_cusip_check_digit                NaN             NaN            -1.0             NaN            -1.0            -1.0             NaN             NaN             NaN   
                   check_digit_sanity                      NaN             NaN             1.0             NaN             1.0             1.0             NaN             NaN             NaN   
securities_load    preflight_csv_files                    99.0             NaN            99.0             NaN            99.0            99.0             NaN             NaN             NaN   
                   load_csv_files                     856547.0             NaN        856547.0             NaN        856547.0        856547.0             NaN           8.208             NaN   
securities_profile securities_count_exact                  1.0             NaN             1.0             NaN             1.0             1.0             NaN           0.020             NaN   
                   securities_totals                       1.0             NaN             1.0             NaN             1.0             1.0             NaN           1.986             NaN   
                   securities_per_year                    25.0             NaN            25.0             NaN            25.0            25.0             NaN           0.069             NaN   
                   securities_duplicates                  50.0             NaN            50.0             NaN            50.0            50.0             NaN           0.433             NaN   
                   securities_check_digit                  NaN             NaN             1.0             NaN             1.0             1.0             NaN             NaN             NaN   
                   repair_check_digit                      NaN             NaN             0.0             NaN          1162.0          1162.0             NaN             NaN             NaN   
asset_build        create_asset_table                      NaN            -1.0            -1.0            -1.0            -1.0            -1.0             NaN             NaN           0.016   
                   insert_underlying                       NaN         42094.0         40593.0         40593.0         40593.0         40593.0             NaN             NaN           1.351   
                   insert_options                          NaN         32315.0         30704.0         26623.0         26623.0             NaN             NaN             NaN

In [25]:
p.close()

Run log: <THESIS_DIR>/src/Clean_solution/Database/Initialisation/pipelines/logs/f13_securities/20261011_201543/steps.jsonl
